# Olist E-commerce — Limpeza e Validação dos Dados
**Autor:** João Victor Azevedo Porto  
**Etapa 02 · Data Cleaning**

Este notebook aplica os tratamentos definidos a partir do Data Discovery: datas, contagens, CEPs, ausentes, duplicatas e relações entre tabelas. Ao final, exporta tabelas tratadas, uma base com uma linha por pedido e relatórios de qualidade.

**Como executar:** abra no Google Colab e use **Ambiente de execução → Executar tudo**. O download é automático pelo KaggleHub, sem montar o Drive. Os arquivos gerados ficam no ambiente do Colab; baixe o ZIP ao terminar.

**Transparência:** este notebook não vem com resultados Olist pré-executados. Os números e o diagnóstico final são calculados durante a execução. O arquivo original e os CSVs baixados não são sobrescritos.

**Verificação do código:** os tratamentos foram testados com dados sintéticos para datas inválidas, contagens fracionárias, centavos, avaliações múltiplas, duplicatas geográficas e preservação dos pedidos. O download e a exportação Parquet precisam ser executados no Colab, junto aos dados reais.


## 1. Regras da limpeza

| Achado do Data Discovery | Tratamento |
|---|---|
| Datas como texto; aprovação fora da inspeção | Converter as oito datas; registrar valores que falharem |
| Contagens como `float` | Converter para `Int64`; valores não inteiros, negativos ou não finitos tornam-se ausentes, com registro |
| Prefixos de CEP numéricos | Ler como texto e padronizar para cinco dígitos |
| Fotos, categorias e comentários ausentes | Preservar desconhecidos; não inventar fotos nem excluir avaliações sem texto |
| Peso zero e dimensões inválidas | Tornar ausentes na camada tratada e registrar os valores originais |
| Parcelas zero e pagamentos zero | Parcela zero torna-se desconhecida; pagamento zero é preservado e sinalizado |
| Prazo de envio extremo, inclusive 2020 | Preservar e sinalizar para revisão; não adivinhar outra data |
| Duplicatas exatas em geolocalização | Remover repetições exatas; gerar tabela separada por prefixo |
| Coordenadas suspeitas | Isolar pontos fora de uma caixa aproximada na consolidação; não considerar a caixa uma fronteira oficial |
| Várias linhas por pedido | Agregar itens e pagamentos antes de juntar; selecionar uma avaliação por regra explícita |

A limpeza preserva todos os pedidos. Registros duvidosos ficam rastreáveis em relatórios; a adequação para cada indicador é definida por flags, não por exclusão indiscriminada.

In [1]:
%pip install -q kagglehub pyarrow

from pathlib import Path
from decimal import Decimal, InvalidOperation
from importlib.metadata import version
import hashlib
import json
import shutil
import platform
import numpy as np
import pandas as pd
import kagglehub
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 30)
print("pandas:", pd.__version__)
print("KaggleHub:", version("kagglehub"))

pandas: 2.2.3
KaggleHub: 1.0.2


## 2. Download e leitura dos dados brutos

Cada CSV é lido como texto para preservar CEPs e valores monetários antes da conversão. `raw` permanece intacto; `clean` recebe os tratamentos. O manifesto registra hashes dos arquivos e versões para identificar exatamente a entrada desta execução.

In [2]:
DATASET = "olistbr/brazilian-ecommerce"
RAW_PATH = Path(kagglehub.dataset_download(DATASET))
FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
missing_files = [name for name in FILES.values() if not (RAW_PATH / name).exists()]
if missing_files:
    raise FileNotFoundError(f"CSV(s) ausente(s): {missing_files}")

raw = {name: pd.read_csv(RAW_PATH / file, dtype="string") for name, file in FILES.items()}
clean = {name: df.copy(deep=True) for name, df in raw.items()}
manifest = {
    "dataset": DATASET, "download_path": str(RAW_PATH),
    "executado_em_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "python": platform.python_version(),
    "pandas": pd.__version__, "numpy": np.__version__,
    "kagglehub": version("kagglehub"), "pyarrow": version("pyarrow"),
    "sha256_csv": {file: hashlib.sha256((RAW_PATH / file).read_bytes()).hexdigest()
                   for file in FILES.values()},
}
display(pd.DataFrame([{"tabela": name, "linhas": len(df), "colunas": len(df.columns)}
                      for name, df in raw.items()]))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


,tabela,linhas,colunas
0,customers,99441,5
1,orders,99441,8
2,items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,translation,71,2


## 3. Auditoria e padronização de texto

Toda substituição de valor inválido registra tabela, coluna, linha de origem, valor anterior e motivo. O índice registrado corresponde à linha de dados do CSV, começando em zero.

Removemos espaços nas bordas de campos estruturados. Comentários de avaliação são preservados, exceto quando contêm apenas espaços. Não removemos acentos de cidades automaticamente, pois isso pode agrupar nomes distintos.

In [4]:
audit_parts = []
changes = []

def log_change(table, column, before, mask, action):
    mask = pd.Series(mask, index=before.index).fillna(False).astype(bool)
    changes.append({"tabela": table, "coluna": column, "acao": action,
                    "valores_afetados": int(mask.sum())})
    if mask.any():
        audit_parts.append(pd.DataFrame({
            "tabela": table, "coluna": column,
            "linha_origem": before.index[mask],
            "valor_original": before.loc[mask].astype("string").to_numpy(),
            "acao": action,
        }))

for name, df in clean.items():
    for col in df.columns:
        before = df[col].copy()
        stripped = before.str.strip()
        after = stripped.mask(stripped.eq(""), pd.NA)
        if col in {"review_comment_title", "review_comment_message"}:
            after = before.mask(stripped.eq(""), pd.NA)
        changed = before.fillna("<NA>").ne(after.fillna("<NA>"))
        log_change(name, col, before, changed, "padronizacao_texto")
        df[col] = after

for name, col in [("customers", "customer_state"), ("sellers", "seller_state"),
                  ("geolocation", "geolocation_state")]:
    before = clean[name][col].copy()
    after = before.str.upper()
    log_change(name, col, before, before.ne(after), "UF_maiuscula")
    clean[name][col] = after

## 4. CEPs e datas

Prefixos devem conter de um a cinco dígitos; completamos zeros à esquerda. Valores fora dessa regra tornam-se ausentes e ficam registrados.

Datas preenchidas que não correspondem ao formato esperado tornam-se `NaT`, sem confundir essas falhas com ausentes já existentes. A data de aprovação está incluída.

In [5]:
for name, col in [("customers", "customer_zip_code_prefix"),
                  ("sellers", "seller_zip_code_prefix"),
                  ("geolocation", "geolocation_zip_code_prefix")]:
    before = clean[name][col].copy()
    invalid = before.notna() & ~before.str.fullmatch(r"\d{1,5}", na=False)
    after = before.mask(invalid).str.zfill(5)
    log_change(name, col, before, invalid, "CEP_invalido_para_ausente")
    log_change(name, col, before, before.notna() & ~invalid & before.ne(after), "CEP_cinco_digitos")
    clean[name][col] = after

DATE_COLS = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
date_report = []
for name, cols in DATE_COLS.items():
    for col in cols:
        before = clean[name][col].copy()
        after = pd.to_datetime(before, format="%Y-%m-%d %H:%M:%S", errors="coerce")
        invalid = before.notna() & after.isna()
        log_change(name, col, before, invalid, "data_invalida_para_NaT")
        clean[name][col] = after
        date_report.append({"tabela": name, "coluna": col,
                            "ausentes_antes": int(before.isna().sum()),
                            "falhas_conversao": int(invalid.sum()),
                            "minimo": after.min(), "maximo": after.max()})
display(pd.DataFrame(date_report))

,tabela,coluna,ausentes_antes,falhas_conversao,minimo,maximo
0,orders,order_purchase_timestamp,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,orders,order_approved_at,160,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,orders,order_delivered_carrier_date,1783,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,orders,order_delivered_customer_date,2965,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,orders,order_estimated_delivery_date,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00
5,items,shipping_limit_date,0,0,2016-09-19 00:15:34,2020-04-09 22:35:08
6,reviews,review_creation_date,0,0,2016-10-02 00:00:00,2018-08-31 00:00:00
7,reviews,review_answer_timestamp,0,0,2016-10-07 18:32:28,2018-10-29 12:27:35


## 5. Contagens, medidas e dinheiro

`Int64` aceita inteiros e ausentes. Frações inesperadas não são arredondadas. Peso e dimensões permanecem decimais, mas precisam ser positivos; fotos e comprimentos textuais aceitam zero como contagem válida.

`payment_installments = 0` passa a ausente porque não informa um número válido de parcelas. O pagamento continua na base. Preço deve ser positivo; frete e pagamento podem ser zero. Valores negativos ou não numéricos tornam-se ausentes.

Valores monetários são convertidos diretamente do texto para **centavos inteiros**, sem passar por float na conversão. Mais de duas casas decimais é uma anomalia, não uma autorização para arredondar silenciosamente. A coluna em reais é mantida para conveniência; somas e reconciliações usam centavos.

In [6]:
def clean_number(table, column, integer=False, minimum=None, maximum=None):
    before = clean[table][column].copy()
    numeric = pd.to_numeric(before, errors="coerce").astype("Float64")
    invalid = before.notna() & (numeric.isna() | ~np.isfinite(numeric))
    if integer:
        invalid |= numeric.mod(1).ne(0).fillna(False)
    if minimum is not None:
        invalid |= numeric.lt(minimum).fillna(False)
    if maximum is not None:
        invalid |= numeric.gt(maximum).fillna(False)
    log_change(table, column, before, invalid, "numero_invalido_para_ausente")
    clean[table][column] = numeric.mask(invalid).astype("Int64" if integer else "Float64")

for col in ["product_photos_qty", "product_name_lenght", "product_description_lenght"]:
    clean_number("products", col, integer=True, minimum=0)
for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    clean_number("products", col, minimum=0)
    before = clean["products"][col].copy()
    zero = before.eq(0)
    log_change("products", col, before, zero, "medida_zero_para_ausente")
    clean["products"][col] = before.mask(zero.fillna(False))
for name, col, lo, hi in [
    ("items", "order_item_id", 1, None),
    ("payments", "payment_sequential", 1, None),
    ("payments", "payment_installments", 1, None),
    ("reviews", "review_score", 1, 5),
]:
    clean_number(name, col, integer=True, minimum=lo, maximum=hi)
for col, lo, hi in [("geolocation_lat", -90, 90), ("geolocation_lng", -180, 180)]:
    clean_number("geolocation", col, minimum=lo, maximum=hi)

MONEY_COLS = {"items": ["price", "freight_value"], "payments": ["payment_value"]}
def to_cents(value):
    if pd.isna(value):
        return pd.NA
    try:
        amount = Decimal(str(value)) * 100
        if not amount.is_finite() or amount != amount.to_integral_value():
            return pd.NA
        result = int(amount)
        return result if abs(result) <= np.iinfo(np.int64).max else pd.NA
    except (InvalidOperation, ValueError, OverflowError):
        return pd.NA

for name, cols in MONEY_COLS.items():
    for col in cols:
        before = clean[name][col].copy()
        cents = before.map(to_cents).astype("Int64")
        invalid = before.notna() & cents.isna()
        invalid |= cents.lt(1 if col == "price" else 0).fillna(False)
        log_change(name, col, before, invalid, "valor_monetario_invalido_para_ausente")
        cents = cents.mask(invalid)
        clean[name][col + "_cents"] = cents
        clean[name][col] = (cents / 100).astype("Float64")
clean["payments"]["flag_payment_zero"] = clean["payments"]["payment_value_cents"].eq(0).fillna(False)

# Corrige apenas os nomes na camada tratada; o mapeamento acompanha a exportação.
RENAME_PRODUCTS = {"product_name_lenght": "product_name_length",
                   "product_description_lenght": "product_description_length"}
clean["products"] = clean["products"].rename(columns=RENAME_PRODUCTS)
manifest["renomeacoes_products"] = RENAME_PRODUCTS

## 6. Geolocalização: duplicatas e consolidação

Removemos duplicatas exatas e registramos as linhas removidas. Depois criamos `geolocation_by_zip`, separada da tabela de observações, com a mediana das coordenadas candidatas por prefixo.

A caixa de latitude −34 a 6 e longitude −74 a −28 é uma **triagem aproximada**. Pontos fora dela não entram na mediana; pontos dentro ainda precisam de validação territorial antes de análises precisas de distância. A coluna `flag_coord_candidate` explicita essa limitação. Prefixos sem coordenadas candidatas continuam na tabela consolidada com coordenadas ausentes.

In [7]:
geo = clean["geolocation"]
# Duplicatas do CSV e repetições que ficaram idênticas após a padronização são auditadas separadamente.
exact_raw = raw["geolocation"].duplicated()
log_change("geolocation", "linha", pd.Series(raw["geolocation"].index.astype(str), index=geo.index),
           exact_raw, "duplicata_exata_removida")
geo = geo.loc[~exact_raw].copy()
normalized_duplicate = geo.duplicated()
log_change("geolocation", "linha", pd.Series(geo.index.astype(str), index=geo.index),
           normalized_duplicate, "duplicata_apos_padronizacao_removida")
geo = geo.loc[~normalized_duplicate].copy()
geo["flag_coord_candidate"] = (
    geo["geolocation_lat"].between(-34, 6) & geo["geolocation_lng"].between(-74, -28)
).fillna(False)
clean["geolocation"] = geo
geo_for_group = geo.copy()
for col in ["geolocation_lat", "geolocation_lng"]:
    geo_for_group[col] = geo_for_group[col].where(geo_for_group["flag_coord_candidate"])
geolocation_by_zip = (
    geo_for_group.dropna(subset=["geolocation_zip_code_prefix"])
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(geolocation_lat=("geolocation_lat", "median"),
         geolocation_lng=("geolocation_lng", "median"),
         observacoes=("flag_coord_candidate", "size"),
         coordenadas_candidatas=("flag_coord_candidate", "sum"),
         cidades_distintas=("geolocation_city", "nunique"),
         UFs_distintas=("geolocation_state", "nunique"))
)
geolocation_by_zip["flag_revisar_geografia"] = (
    geolocation_by_zip["coordenadas_candidatas"].eq(0)
    | geolocation_by_zip["UFs_distintas"].gt(1)
)
print("Duplicatas exatas removidas:", int(exact_raw.sum()))
print("Coordenadas fora da triagem ou ausentes:", int((~geo["flag_coord_candidate"]).sum()))
display(geolocation_by_zip.head())

Duplicatas exatas removidas: 261831
Coordenadas fora da triagem ou ausentes: 27


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,observacoes,coordenadas_candidatas,cidades_distintas,UFs_distintas,flag_revisar_geografia
0,01001,-23.549951,-46.634027,11,11,2,1,False
1,01002,-23.548228,-46.635247,6,6,2,1,False
2,01003,-23.548977,-46.635313,11,11,2,1,False
3,01004,-23.54955,-46.634771,14,14,2,1,False
4,01005,-23.549763,-46.6361,13,13,2,1,False


## 7. Chaves, relações e categorias

Não removemos linhas apenas porque uma chave se repete. Chaves primárias esperadas com nulos ou duplicatas interrompem a montagem da base analítica para impedir uma junção incorreta. Avaliações são tratadas separadamente.

Ausência de tradução e de cobertura geográfica é sinalizada; não apagamos produtos ou clientes por isso. Categoria ausente continua ausente. Para apresentação, usamos uma coluna adicional com o rótulo `nao_informada`.

In [8]:
PRIMARY_KEYS = {"orders": ["order_id"], "customers": ["customer_id"],
                "items": ["order_id", "order_item_id"],
                "payments": ["order_id", "payment_sequential"],
                "products": ["product_id"], "sellers": ["seller_id"],
                "translation": ["product_category_name"]}
key_report = pd.DataFrame([
    {"tabela": name, "chave": " + ".join(keys),
     "linhas_com_chave_nula": int(clean[name][keys].isna().any(axis=1).sum()),
     "duplicatas_excedentes": int(clean[name].duplicated(keys).sum())}
    for name, keys in PRIMARY_KEYS.items()
])
RELATIONS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]
relation_rows = []
for child, ck, parent, pk in RELATIONS:
    source = clean[child][ck]
    orphan = source.notna() & ~source.isin(clean[parent][pk].dropna())
    relation_rows.append({"origem": child, "coluna": ck, "destino": parent,
                          "nulos": int(source.isna().sum()),
                          "linhas_orfas": int(orphan.sum()),
                          "chaves_orfas": int(source[orphan].nunique())})
relation_report = pd.DataFrame(relation_rows)
display(key_report)
display(relation_report)
if key_report[["linhas_com_chave_nula", "duplicatas_excedentes"]].to_numpy().sum():
    raise ValueError("Chaves inválidas: revise key_report e a auditoria antes de continuar.")
# As seis primeiras relações sustentam o modelo de pedidos, itens e pagamentos.
if relation_report.iloc[:6][["nulos", "linhas_orfas"]].to_numpy().sum():
    raise ValueError("Relações centrais inválidas: inspecione relation_report antes de continuar.")

products = clean["products"]
products["flag_category_missing"] = products["product_category_name"].isna()
products["category_label"] = products["product_category_name"].fillna("nao_informada")
products["flag_translation_missing"] = (
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(clean["translation"]["product_category_name"])
)

,tabela,chave,linhas_com_chave_nula,duplicatas_excedentes
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,items,order_id + order_item_id,0,0
3,payments,order_id + payment_sequential,0,0
4,products,product_id,0,0
5,sellers,seller_id,0,0
6,translation,product_category_name,0,0


,origem,coluna,destino,nulos,linhas_orfas,chaves_orfas
0,orders,customer_id,customers,0,0,0
1,items,order_id,orders,0,0,0
2,items,product_id,products,0,0,0
3,items,seller_id,sellers,0,0,0
4,payments,order_id,orders,0,0,0
5,reviews,order_id,orders,0,0,0
6,products,product_category_name,translation,610,13,2
7,customers,customer_zip_code_prefix,geolocation,0,278,157
8,sellers,seller_zip_code_prefix,geolocation,0,7,7


## 8. Datas suspeitas e elegibilidade para entrega

Preservamos datas cronologicamente suspeitas e adicionamos flags. Um prazo de envio superior a 365 dias após a compra é uma regra de triagem configurável, não prova de erro.

Para comparar entrega e previsão, usamos o dia do calendário, evitando marcar como atrasada uma entrega no mesmo dia de uma previsão armazenada à meia-noite. Pedidos sem os campos necessários ficam com métricas ausentes, não com atraso igual a zero.

In [9]:
orders = clean["orders"]
sequence = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
]
orders["flag_date_sequence"] = False
for start, end in sequence:
    orders["flag_date_sequence"] |= orders[start].notna() & orders[end].notna() & orders[end].lt(orders[start])
orders["flag_delivered_without_date"] = (
    orders["order_status"].eq("delivered") & orders["order_delivered_customer_date"].isna()
)
orders["flag_date_without_delivered_status"] = (
    orders["order_delivered_customer_date"].notna() & ~orders["order_status"].eq("delivered")
).fillna(False)
valid_delivery = (
    orders["order_status"].eq("delivered")
    & orders[["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]].notna().all(axis=1)
    & ~orders["flag_date_sequence"]
).fillna(False)
orders["eligible_delivery_analysis"] = valid_delivery
orders["delivery_days"] = (
    (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
).where(valid_delivery)
orders["delay_days"] = (
    orders["order_delivered_customer_date"].dt.normalize()
    - orders["order_estimated_delivery_date"].dt.normalize()
).dt.days.where(valid_delivery).astype("Int64")
orders["is_late"] = orders["delay_days"].gt(0).astype("boolean").where(valid_delivery, pd.NA)

SHIPPING_REVIEW_DAYS = 365
items_context = clean["items"].merge(
    orders[["order_id", "order_purchase_timestamp"]], on="order_id", how="left", validate="many_to_one"
)
interval = (items_context["shipping_limit_date"] - items_context["order_purchase_timestamp"]).dt.total_seconds() / 86400
clean["items"]["flag_shipping_review"] = (interval.lt(0) | interval.gt(SHIPPING_REVIEW_DAYS)).to_numpy()
manifest["limiar_revisao_prazo_dias"] = SHIPPING_REVIEW_DAYS
shipping_review = items_context.loc[interval.lt(0) | interval.gt(SHIPPING_REVIEW_DAYS)].copy()
shipping_review["days_purchase_to_limit"] = interval.loc[shipping_review.index]
display(shipping_review.sort_values("shipping_limit_date", ascending=False).head(10))

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,price_cents,freight_value_cents,order_purchase_timestamp,days_purchase_to_limit
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44,9999,6144,2017-05-23 22:28:36,1052.004537
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44,9999,6144,2017-05-23 22:28:36,1052.004537
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66,6999,1466,2017-03-16 02:30:51,1056.041667
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.7,7599,1470,2017-03-14 19:23:22,1056.041667


## 9. Uma linha por pedido, sem multiplicar valores

Itens e pagamentos são agregados separadamente. Se qualquer componente monetário do grupo estiver ausente, o total fica ausente: uma soma parcial não deve parecer um total completo.

Para avaliações, selecionamos a resposta mais recente por pedido; em empate, usamos data de criação, `review_id` e posição na fonte como desempates. Mantemos todas as avaliações em `clean["reviews"]` e registramos quantas existiam por pedido. A regra é analítica, não uma correção da fonte.

A diferença entre pagamentos e itens com frete é apenas uma reconciliação. Não inferimos lucro, receita líquida ou estorno com esses campos.

In [10]:
def complete_sum(series):
    return series.sum(min_count=len(series))

items_by_order = clean["items"].groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "size"),
    merchandise_cents=("price_cents", complete_sum),
    freight_cents=("freight_value_cents", complete_sum),
    flag_shipping_review=("flag_shipping_review", "max"),
)
payments_by_order = clean["payments"].groupby("order_id", as_index=False).agg(
    payment_count=("payment_sequential", "size"),
    paid_cents=("payment_value_cents", complete_sum),
    flag_payment_zero=("flag_payment_zero", "max"),
)
review_source = clean["reviews"].copy()
review_source["source_row"] = review_source.index
review_counts = review_source.groupby("order_id").size().rename("review_count")
review_one = (
    review_source.sort_values(
        ["order_id", "review_answer_timestamp", "review_creation_date", "review_id", "source_row"],
        ascending=[True, False, False, True, True], na_position="last", kind="stable"
    ).drop_duplicates("order_id", keep="first")
)
review_one = review_one.merge(review_counts, on="order_id", how="left", validate="one_to_one")
orders_analysis = (
    clean["orders"]
    .merge(clean["customers"], on="customer_id", how="left", validate="many_to_one")
    .merge(items_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(payments_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(review_one[["order_id", "review_id", "review_score", "review_answer_timestamp", "review_count"]],
           on="order_id", how="left", validate="one_to_one")
)
for col in ["item_count", "payment_count", "review_count"]:
    orders_analysis[col] = orders_analysis[col].fillna(0).astype("Int64")
for col in ["merchandise_cents", "freight_cents", "paid_cents"]:
    orders_analysis[col] = orders_analysis[col].astype("Int64")
orders_analysis["expected_cents"] = orders_analysis["merchandise_cents"] + orders_analysis["freight_cents"]
orders_analysis["payment_difference_cents"] = orders_analysis["paid_cents"] - orders_analysis["expected_cents"]
orders_analysis["flag_payment_mismatch"] = orders_analysis["payment_difference_cents"].ne(0).astype("boolean")
orders_analysis["eligible_sales_analysis"] = (
    orders_analysis["order_status"].eq("delivered")
    & orders_analysis["order_purchase_timestamp"].notna()
    & orders_analysis["merchandise_cents"].notna()
    & orders_analysis["item_count"].gt(0)
).fillna(False)
# Esta regra define o recorte de vendas entregues; todos os outros pedidos continuam na base.
orders_analysis["eligible_delivery_review_analysis"] = (
    orders_analysis["eligible_delivery_analysis"] & orders_analysis["review_score"].notna()
)
display(orders_analysis.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_date_sequence,flag_delivered_without_date,flag_date_without_delivered_status,eligible_delivery_analysis,delivery_days,delay_days,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,merchandise_cents,freight_cents,flag_shipping_review,payment_count,paid_cents,flag_payment_zero,review_id,review_score,review_answer_timestamp,review_count,expected_cents,payment_difference_cents,flag_payment_mismatch,eligible_sales_analysis,eligible_delivery_review_analysis
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,True,8.436574,-8,False,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,1,2999,872,False,3,3871,False,a54f0611adc9ed256b57ede6b6eb5114,4,2017-10-12 03:43:48,1,3871,0,False,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,True,13.782037,-6,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,11870,2276,False,1,14146,False,8d5266042046a06655c8db133d120ba5,4,2018-08-08 18:37:50,1,14146,0,False,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,True,9.394213,-18,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,15990,1922,False,1,17912,False,e73b67b67587f7644d5bd1a52deb1b01,5,2018-08-22 19:07:58,1,17912,0,False,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,True,13.208750,-13,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,4500,2720,False,1,7220,False,359d03e676b3c069f62cadba8dd3f6e8,5,2017-12-05 19:21:58,1,7220,0,False,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,True,2.873877,-10,False,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,1,1990,872,False,1,2862,False,e50934924e227544ba8246aeb3770dd4,5,2018-02-18 13:02:51,1,2862,0,False,True,True


## 10. Validação final e pendências

Os controles verificam preservação dos pedidos, tipos, chaves e totais após as junções. As pendências não são escondidas: os relatórios mostram quais registros ainda exigem atenção e quais podem entrar nas análises definidas.

Somas de todos os itens brutos tratados podem incluir componentes válidos de pedidos com outro componente ausente. Por isso, a verificação abaixo compara os totais da base final com os totais completos já agregados por pedido.

In [11]:
assert len(orders_analysis) == len(raw["orders"]), "A junção alterou a quantidade de pedidos."
assert orders_analysis["order_id"].is_unique
assert orders_analysis["order_id"].notna().all()
for name, cols in DATE_COLS.items():
    for col in cols:
        assert pd.api.types.is_datetime64_any_dtype(clean[name][col])
for col in ["product_photos_qty", "product_name_length", "product_description_length"]:
    assert str(clean["products"][col].dtype) == "Int64"
for name, col in [("customers", "customer_zip_code_prefix"), ("sellers", "seller_zip_code_prefix"),
                  ("geolocation", "geolocation_zip_code_prefix")]:
    assert clean[name][col].dropna().str.fullmatch(r"\d{5}").all()
for col, source in [("merchandise_cents", items_by_order), ("freight_cents", items_by_order),
                    ("paid_cents", payments_by_order)]:
    assert orders_analysis[col].sum() == source[col].sum(), f"Total alterado: {col}"
assert geolocation_by_zip["geolocation_zip_code_prefix"].is_unique
for name, df in clean.items():
    if name != "geolocation":
        assert len(df) == len(raw[name]), f"Linhas removidas inesperadamente em {name}"

changes_report = pd.DataFrame(changes)
audit = (pd.concat(audit_parts, ignore_index=True) if audit_parts else
         pd.DataFrame(columns=["tabela", "coluna", "linha_origem", "valor_original", "acao"]))
quality = pd.DataFrame([
    {"indicador": "Pedidos preservados", "quantidade": len(orders_analysis)},
    {"indicador": "Pedidos com sequência temporal suspeita", "quantidade": int(orders["flag_date_sequence"].sum())},
    {"indicador": "Entregues sem data de entrega", "quantidade": int(orders["flag_delivered_without_date"].sum())},
    {"indicador": "Data de entrega com status diferente de delivered", "quantidade": int(orders["flag_date_without_delivered_status"].sum())},
    {"indicador": "Itens com prazo de envio para revisão", "quantidade": len(shipping_review)},
    {"indicador": "Pedidos sem itens", "quantidade": int(orders_analysis["item_count"].eq(0).sum())},
    {"indicador": "Pedidos sem pagamentos", "quantidade": int(orders_analysis["payment_count"].eq(0).sum())},
    {"indicador": "Pedidos com várias avaliações", "quantidade": int(orders_analysis["review_count"].gt(1).sum())},
    {"indicador": "Diferenças de pagamento conhecidas", "quantidade": int(orders_analysis["flag_payment_mismatch"].fillna(False).sum())},
    {"indicador": "Reconciliações indisponíveis", "quantidade": int(orders_analysis["payment_difference_cents"].isna().sum())},
    {"indicador": "Pedidos elegíveis para vendas entregues", "quantidade": int(orders_analysis["eligible_sales_analysis"].sum())},
    {"indicador": "Pedidos elegíveis para entrega e avaliação", "quantidade": int(orders_analysis["eligible_delivery_review_analysis"].sum())},
])
missing_after = pd.DataFrame([
    {"tabela": name, "coluna": col, "tipo": str(df[col].dtype),
     "ausentes": int(df[col].isna().sum()), "percentual": round(df[col].isna().mean() * 100, 2)}
    for name, df in clean.items() for col in df.columns
])
display(changes_report.query("valores_afetados > 0"))
display(quality)
print("Controles estruturais concluídos. Consulte as pendências antes de escolher o recorte analítico.")

,tabela,coluna,acao,valores_afetados
73,products,product_weight_g,medida_zero_para_ausente,4
82,payments,payment_installments,numero_invalido_para_ausente,2
89,geolocation,linha,duplicata_exata_removida,261831


,indicador,quantidade
0,Pedidos preservados,99441
1,Pedidos com sequência temporal suspeita,1382
2,Entregues sem data de entrega,8
3,Data de entrega com status diferente de delivered,6
4,Itens com prazo de envio para revisão,4
5,Pedidos sem itens,775
6,Pedidos sem pagamentos,1
7,Pedidos com várias avaliações,547
8,Diferenças de pagamento conhecidas,576
9,Reconciliações indisponíveis,776


Controles estruturais concluídos. Consulte as pendências antes de escolher o recorte analítico.


## 11. Exportação para a próxima etapa

O ZIP contém as nove tabelas tratadas em **Parquet**, que preserva tipos e ausentes, além das bases auxiliares, auditorias em CSV, manifesto e dicionário de tipos. Os arquivos ficam somente no ambiente local do Colab até você baixá-los.





In [12]:
OUTPUT = Path("/content/olist_cleaning_output") if Path("/content").exists() else Path.cwd() / "olist_cleaning_output"
for folder in ["clean", "analytical", "reports"]:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)
for name, df in clean.items():
    df.to_parquet(OUTPUT / "clean" / f"{name}.parquet", index=False)
for name, df in {"orders_analysis": orders_analysis, "geolocation_by_zip": geolocation_by_zip,
                 "review_one_per_order": review_one}.items():
    df.to_parquet(OUTPUT / "analytical" / f"{name}.parquet", index=False)
reports = {"changes": changes_report, "audit_values": audit, "quality": quality,
           "keys": key_report, "relations": relation_report,
           "dates": pd.DataFrame(date_report), "schema_and_missing": missing_after,
           "shipping_review": shipping_review}
for name, df in reports.items():
    df.to_csv(OUTPUT / "reports" / f"{name}.csv", index=False, encoding="utf-8-sig")
manifest["regras"] = {
    "vendas": "Pedidos delivered, com compra válida, itens e valor de mercadorias completo",
    "avaliacao": "Resposta mais recente; desempate por criação, review_id e linha da fonte",
    "dinheiro": "Centavos inteiros; totais incompletos permanecem ausentes",
    "geografia": "Mediana por prefixo dentro de caixa aproximada; validação territorial pendente",
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
summary_lines = ["# Resultado da limpeza Olist", "", "Indicadores desta execução:", ""]
summary_lines += [f"- {row.indicador}: {row.quantidade:,}" for row in quality.itertuples(index=False)]
summary_lines += ["", "Consulte reports/ para alterações e pendências. Ausentes não foram preenchidos por suposição.",
                  "Próxima etapa: evolução mensal das vendas entregues e associação entre atraso e nota."]
(OUTPUT / "README.md").write_text(chr(10).join(summary_lines), encoding="utf-8")
zip_path = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
print("Arquivo pronto para baixar:", zip_path)

Arquivo pronto para baixar: /content/olist_cleaning_output.zip


### Download opcional

Execute a célula abaixo para baixar o pacote. O notebook termina com os dados exportados, mas a cópia no Colab é temporária: mantenha o ZIP junto ao projeto para a próxima etapa.

In [13]:
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>